# pycograd — compiling the DSL to Apple Silicon (Metal / MPS)

This is the **Apple-GPU** companion to [`pycograd_compile_torch_demo`](pycograd_compile_torch_demo.ipynb): the same ambient-weights DSL — `with params{...} as weights`, a `$ |> ...` pipeline, and `weights.grad(objective)` — but every net is **trained and run on the Metal Performance Shaders (MPS) backend**, PyTorch's Apple-Silicon GPU device. Two seams do it:

* **`weights.grad(objective, backend="mps", jit=True)`** differentiates the same ambient objective with PyTorch's autodiff *on the GPU*, and compiles that gradient **once** (the net is traced a single time, then reused every step).
* **`compile_to(forward, "mps")`** runs the same `forward` over PyTorch tensors that live on the Metal device.

**A note on precision.** MPS does not support float64, so the `mps` backend computes pycograd's float64 default **in float32**; gradients then agree with the numpy tape to single-precision tolerance (~1e-6) rather than ~1e-12. (An explicit `with dtype("float16" | "bf16"):` block is honored as-is — those are MPS-native.)

The setup cell below falls back to the CPU `torch` backend when no MPS device is present, so this notebook executes anywhere; on an Apple-Silicon Mac it runs on the GPU.

In [1]:
%load_ext pycograd

## Setup: losses and data

There is no op library to import — pycograd differentiates raw numpy, so the loss and activations are just numpy used as pipe stages. The models below are all rank-2 matmuls, so they compile cleanly on every backend.

In [2]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # quiet TensorFlow's C++ logging
import warnings; warnings.filterwarnings("ignore")

import numpy as np
from pycograd import compile_to, get_backend, frozen

# --- losses & activations: ordinary numpy, used as pipe stages -----------------
def softmax_ce(logits, onehot):
    z = logits - np.max(logits, axis=1, keepdims=True)          # stabilize
    logp = z - np.log(np.sum(np.exp(z), axis=1, keepdims=True)) # log-softmax
    return -np.mean(np.sum(onehot * logp, axis=1))

def relu(z):     return np.maximum(0.0, z)
def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))

# --- data: three Gaussian blobs, 3-way classification --------------------------
rng = np.random.default_rng(0)
m = 40
centers = np.array([[2.0, 2.0], [-2.0, 2.0], [0.0, -2.5]])
X = np.vstack([rng.normal(c, 0.5, (m, 2)) for c in centers])
labels = np.repeat(np.arange(3), m)
Y = np.eye(3)[labels]

# --- this notebook runs on Apple's Metal GPU (MPS), with a CPU fallback --------
import torch
import logging; logging.getLogger("torch._inductor.utils").setLevel(logging.ERROR)  # quiet a benign GPU perf note
MPS_OK = bool(getattr(torch.backends, "mps", None)) and torch.backends.mps.is_available()
BACKEND = "mps" if MPS_OK else "torch"
print("MPS available:", MPS_OK, "— training on backend:", repr(BACKEND))

def train(weights, objective, steps, lr):
    """Train on the selected backend (MPS GPU, or CPU torch): every step's gradient comes from PyTorch's own autodiff, and
    jit=True compiles that gradient ONCE (the net is traced a single time, not per step)."""
    first = last = None
    for _ in range(steps):
        value, grads = weights.grad(objective, backend=BACKEND, jit=True)
        first = float(value) if first is None else first
        last = float(value)
        weights.step(grads, lr)                 # plain in-place SGD on the numpy weights
    return first, last

def accuracy(logits):
    return float(np.mean(np.argmax(np.asarray(logits), axis=1) == labels))

MPS available: True — training on backend: 'mps'


## Example 1 — a linear softmax classifier, trained on PyTorch

The forward is one pipe stage, `$ @ w + b`; `weights.grad(objective, backend="torch", jit=True)` differentiates it with PyTorch and compiles the step once. Nothing about the model definition changed from the numpy demo — only where the gradient comes from.

In [3]:
with params{
    w = np.zeros((2, 3))
    b = np.zeros(3)
} as weights:
    forward = $ |> $ @ w + b                       # logits; the loss does the softmax
    objective = lambda: forward(X) |> softmax_ce($, Y)
    first, last = train(weights, objective, 200, 0.5)
    logits = forward(X)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", accuracy(logits))

loss 1.0986 -> 0.0040
train accuracy: 1.0


## Run the forward on PyTorch's own tensors

`compile_to(forward, "torch")` turns the **same** DSL `forward` into a callable over PyTorch tensors (the ambient weights are lifted onto the backend automatically).

In [4]:
with params{
    w = np.zeros((2, 3))
    b = np.zeros(3)
} as weights:
    forward = $ |> $ @ w + b
    be = get_backend(BACKEND)
    run = compile_to(forward, BACKEND)             # a forward on PyTorch's own tensors
    out = run(be.const(X))                         # pass an PyTorch tensor in, get one out
    native = np.asarray(be.to_numpy(out))
    ref = forward(X)                               # the same forward in numpy

print("compile_to returned a", type(out).__module__.split(".")[0],
      "tensor of shape", tuple(native.shape))
print("matches numpy:", np.allclose(native, ref))

compile_to returned a torch tensor of shape (120, 3)
matches numpy: True


## Gradients agree with the numpy tape

Calling `weights.grad(objective)` with no backend runs pycograd's own numpy tape; with `backend="torch"` the gradient comes from PyTorch. They match to tight tolerance — and since the tape is finite-difference-checked, that validates the compile path end-to-end.

In [5]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = np.zeros(16)
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> relu |> $ @ w2 + b2
    objective = lambda: forward(X) |> softmax_ce($, Y)
    v_np, g_np = weights.grad(objective)                    # pycograd's numpy tape
    v_be, g_be = weights.grad(objective, backend=BACKEND)   # PyTorch's own autodiff
    worst = max(np.max(np.abs(np.asarray(g_be[k]) - np.asarray(g_np[k]))) for k in weights)

print("value  numpy=%.6f  %s=%.6f" % (float(v_np), BACKEND, float(v_be)))
print("max |grad_%s - grad_numpy| over all weights: %.1e" % (BACKEND, worst))

value  numpy=1.514232  mps=1.514232
max |grad_mps - grad_numpy| over all weights: 6.1e-08


## Example 2 — a 2-layer MLP, as one pipeline

`relu` is just a pipe stage. Same DSL, same training call — a deeper net.

In [6]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = np.zeros(16)
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> relu |> $ @ w2 + b2
    objective = lambda: forward(X) |> softmax_ce($, Y)
    first, last = train(weights, objective, 300, 0.5)
    logits = forward(X)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", accuracy(logits))

loss 1.8837 -> 0.0007
train accuracy: 1.0


## A frozen parameter

`frozen[...]` holds a weight fixed: its gradient comes back `None` from PyTorch's autodiff (same semantics as the numpy tape) and `weights.step` skips it.

In [7]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = frozen[np.zeros(16)]                       # held fixed: gradient comes back None
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> relu |> $ @ w2 + b2
    objective = lambda: forward(X) |> softmax_ce($, Y)
    _, g_be = weights.grad(objective, backend=BACKEND)
    first, last = train(weights, objective, 300, 0.5)
    logits = forward(X)

print("frozen b1 gradient:", g_be["b1"])            # None -> excluded from PyTorch's autodiff
print("loss %.4f -> %.4f  |  b1 stayed all-zero: %s" % (first, last, bool(np.all(weights["b1"].value == 0))))
print("train accuracy:", accuracy(logits))

frozen b1 gradient: None
loss 1.5022 -> 0.0008  |  b1 stayed all-zero: True
train accuracy: 1.0


## Ship it: a real `nn.Module`, then TorchScript / ONNX

`weights.to_torch_module(forward)` wraps the DSL net as a genuine `torch.nn.Module` — trainable leaves become `Parameter`s, frozen leaves become buffers. The training above ran on MPS; the wrapped module is a standard (CPU-dtype) `nn.Module` you can train with any torch optimizer (`loss.backward()`) or move to the GPU with `module.to("mps")`. TorchScript / ONNX capture its forward graph into a file that runs with **no pycograd dependency**. Build, run, and trace it inside `with weights:` (the forward reads the injected proxies); the exported artifact is then standalone.

In [8]:
import os, tempfile, torch
from pycograd import export_torchscript, export_onnx

with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = np.zeros(16)
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> relu |> $ @ w2 + b2
    objective = lambda: forward(X) |> softmax_ce($, Y)
    train(weights, objective, 200, 0.5)                 # train on torch

    module = weights.to_torch_module(forward)           # a real, trainable nn.Module
    xb = torch.as_tensor(np.asarray(X))
    print("module logits match numpy:", np.allclose(module(xb).detach().numpy(), forward(X)))
    print("parameters:", [tuple(p.shape) for p in module.parameters()])

    d = tempfile.mkdtemp()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                 # quiet torch.jit's constant-trace notes
        export_torchscript(module, (xb,), os.path.join(d, "clf.pt"))
        reloaded = torch.jit.load(os.path.join(d, "clf.pt"))   # no pycograd needed to run this
    print("TorchScript reload matches:",
          np.allclose(reloaded(xb).detach().numpy(), module(xb).detach().numpy()))

    try:
        import onnxruntime as ort
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            export_onnx(module, (xb,), os.path.join(d, "clf.onnx"),
                        input_names=["X"], opset_version=17)
        out = ort.InferenceSession(os.path.join(d, "clf.onnx")).run(None, {"X": np.asarray(X)})[0]
        print("ONNX runtime matches:", np.allclose(out, module(xb).detach().numpy(), atol=1e-6))
    except ImportError:
        print("onnxruntime not installed — export_onnx still writes a standalone .onnx.")

module logits match numpy: True
parameters: [(2, 16), (16,), (16, 3), (3,)]
TorchScript reload matches: True
onnxruntime not installed — export_onnx still writes a standalone .onnx.


## More

Richer models — highway nets, self-attention, a full Transformer encoder block — are written in exactly this DSL in [`pycograd_demo`](pycograd_demo.ipynb), and each trains on a framework the same way: add `backend=`/`jit=` to `weights.grad`. The cross-backend conformance suite checks forward + gradient parity for every installed framework:

```
python -m pytest test/test_compile.py
```